### Import Dependencies

In [ ]:
import openai
import instructor
from pydantic import BaseModel, Field

from qdrant_client import QdrantClient

from dotenv import load_dotenv

load_dotenv("../../.env")

### Mock Example

In [ ]:
prompt ="""You are a helpful assistant.
Return an answer to the question.
Question: what is your name?"""

In [ ]:
response = openai.chat.completions.create(
    model="gpt-5.4-nano",
    messages=[
        {"role": "system", "content": prompt},
    ],
    reasoning_effort="none",
)

print(response.choices[0].message.content)


In [ ]:
response

### Add Instructor (Structured Outputs)

In [ ]:
client = instructor.from_provider(
  "openai/gpt-5.4-nano",
  mode=instructor.Mode.RESPONSES_TOOLS
)

In [ ]:
class Answer(BaseModel):
  answer: str = Field(description="The answer to the question")
  

In [ ]:
response = client.create(
  messages=[
    {"role": "user", "content": prompt}
  ],
  reasoning={"effort": "none"},
  response_model=Answer
)

In [ ]:
response

In [ ]:
response, raw_response = client.create_with_completion(
    messages=[{"role": "user", "content": prompt}],
    reasoning={"effort": "none"},
    response_model=Answer,
)

In [ ]:
response


In [ ]:
raw_response

In [ ]:
class AnswerWithReasoning(BaseModel):
  reasoning: str = Field(description="Reasoning for the answer")
  answer: str = Field(description="Answer to the question") 

In [ ]:
response, raw_response = client.create_with_completion(
    messages=[{"role": "user", "content": prompt}],
    reasoning={"effort": "none"},
    response_model=AnswerWithReasoning,
)

In [ ]:
response

In [ ]:
raw_response

### RAG Pipeline

In [ ]:
class RAGGenerationResponse(BaseModel):
  answer: str = Field(description="Answer to the question")

In [ ]:
qdrant_client = QdrantClient(url="http://localhost:6333")

embedding_model = "text-embedding-3-small"


def get_embedding(text, model=embedding_model):
    response = openai.embeddings.create(input=text, model=model)
    return response.data[0].embedding


def retrieve_data(query, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01", query=query_embedding, limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        if not result.payload:
            raise ValueError("No payload found in Qdrant ScoredPoint")
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings,
    }


def process_context(context):
    formatted_context = ""

    for id, chunk, rating in zip(
        context["retrieved_context_ids"],
        context["retrieved_context"],
        context["retrieved_context_ratings"],
    ):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}
"""
    return prompt


answer_gen_model = "gpt-5.4-nano"


def generate_answer(prompt):
    response, _raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt},
        ],
        reasoning={"effort": "none"},
        response_model=RAGGenerationResponse,
    )

    return response


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    response = generate_answer(prompt)

    final_answer = {
        "data_object": response,
        "answer": response.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
    }

    return final_answer

In [ ]:
output = rag_pipeline("Any USB chargeable speakers?", top_k=10)

In [ ]:
output